In [1]:
import glob
import os
import time
import warnings
from urllib.error import URLError

import math
import geopandas as gpd
import matplotlib.pyplot as plt
import nivapy3 as nivapy
import numpy as np
import pandas as pd
import utils
from tqdm.notebook import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter

warnings.simplefilter("ignore")
plt.style.use("ggplot")

# Get gridded daily catchment data

Use the GTS API to get modelled water discharge and met data for each catchment, for the period 1980-present. 1 km gridded data, aggregated (averaged) over the catchment.

Data comes from NVE's GridTimeSeries API (http://api.nve.no/doc/gridtimeseries-data-gts/). This delivers time series data for a single point. Latest status for parameters: https://gts.nve.no/HealthCheck/Detailed

Parameters of potential interest for now:
- rr: døgnnedbør, mm/day. From seNorge gridded precipitation dataset
- tm: temperatur, degC. From SeNorge gridded temperature dataset
- gwb_q: avrenning (runoff), mm/day. Gridded Water Balance (GWB) model driven by seNorge precip and temp.

Precip and temp are produced by spatial interpolation of station observations, supplemented by climate-model information during interpolation. Result is a 1 km gridded dataset covering Norway from 1957 onwards. Runoff is produced by the GWB1 model, a distributed version of NVE's HBV hydrological model. Inputs are gridded precip and temp from seNorge. The model simulates variables such as soil moisture, groundwater conditions and runoff. gwb_q is the resulting runoff estimate.

Here, we download point data and then average over the catchment. For small catchments (< e.g. 10 km2), all points are downloaded. For larger catchments, we take a random sample and average that, where the sample size is estimated to give an error of around 3-4%. Previous experiments by James (part of the water temp project with Jan Erik) suggested a random sample of 200-300 has essentially zero error compared to downloading all grid points for large catchments, and the sampling approach taken here approaches just over 200 samples for large catchments.

**N.B. There two 2-day gaps in downloaded modelled discharge (spotted during QC). Not fixed here, but when the data are processed elsewhere**

End result for Vannmiljø TOC catchments: that successfully downloaded met data for n-1 sites and Q data for n-4 sites. i.e. **5 sites overall that don't have data for one or the other of Q or temp/precip**. These include the data-rich QUANTOM site Utsjoki (all in Finland) and Pasvikelva, which also has lots of data.

## Read catchment boundaries

In [2]:
# Catchment boundaries
cat_fpath = r"/home/jovyan/shared/common/norsink/spatial/monitored_chem_catchments/catchments_vm-rivs-quant-oc_v3-full.gpkg"

# Read catchments
cat_gdf = gpd.read_file(cat_fpath)

cat_gdf.head()

# cat_gdf_test = cat_gdf.iloc[:30].copy()
# cat_gdf_test

,site_code,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,av_freq,area_km2,geometry
0,001-119928,11.653745,59.476520,119928,Oppstrøms Bommen RA,2011,2025,14.0,14,1.00,1008.505005,"POLYGON ((306680 6664180, 306760 6664180, 3067..."
1,001-119929,11.654461,59.472300,119929,Nedstrøms Bommen RA,2011,2025,14.0,14,1.00,1009.065359,"POLYGON ((306680 6664180, 306760 6664180, 3067..."
2,001-30653,11.437438,59.070299,30653,YSTE YSTEHEDEBEKKEN,1987,1987,1.0,4,4.00,6.702626,"POLYGON ((298520 6555300, 298640 6555300, 2986..."
3,001-30683,11.295421,59.121185,30683,"Unnebergbekken, UNNE",1987,2016,29.1,5,0.17,15.372746,"POLYGON ((290660 6567260, 290680 6567260, 2906..."
4,001-30687,11.528412,58.987892,30687,ØRB2 ØRBEKKEN,1991,1991,1.0,4,4.00,42.290760,"POLYGON ((305560 6548480, 305560 6548500, 3055..."


## Get data

Modelled runoff time series were extracted from NVE’s 1 km GridTimeSeries dataset using a variable sampling strategy designed to balance computational efficiency and precision. For each catchment containing N grid cells, the required sample size n was estimated from an assumed coefficient of variation (CV = 0.5) and a target relative standard error of 3.5% for the catchment mean runoff. An initial sample size was calculated using n0 = (CV/RSE)^2, followed by application of the finite population correction:

n = (N * n0) / (N + n0 − 1)

The resulting sample size was bounded between a minimum of 10 and a maximum of 250 cells. A minimum sample size of 15 grid cells was requested for small catchments, although all available cells were used whenever fewer than 15 cells were present. For larger catchments, sample size increased smoothly with catchment area and approached an asymptotic maximum of approximately 200 cells. This approach avoids arbitrary catchment-size thresholds while providing a consistent level of sampling precision and substantially reducing the number of API requests required for large catchments.

In [3]:
def get_sample_size(
    area_km2,
    cv=0.5,
    target_rse=0.035,
    min_samples=10,
    max_samples=250,
):
    """
    Determine sample size from catchment area assuming 1 km² grid cells.

    Parameters
    ----------
    area_km2: float. Used to calculate number of grid cells in catchment.
    cv : float. Assumed coefficient of variation.
    target_rse : float. Desired relative standard error.
    min_samples : int. Minimum sample size (i.e. lower cap)
    max_samples : int. Maximum sample size (i.e. cap)

    Returns
    -------
    int. Recommended sample size
    """

    if area_km2 < min_samples:
        return min_samples  # effectively "use all cells"

    N = math.ceil(area_km2)

    n0 = (cv / target_rse) ** 2
    n = math.ceil((N * n0) / (N + n0 - 1))

    return min(max_samples, n)


def process_catchment(cat_id, stn_gdf):
    csv_path = os.path.join(temp_dir, f"{cat_id}.csv")

    if os.path.isfile(csv_path):
        return# f"{cat_id}: already exists"

    retries = 0

    while retries < max_retries:
        try:
            cat_area = stn_gdf.iloc[0][cat_area_col]

            # determine sample size for this catchment
            n_samp = get_sample_size(cat_area)
            # print("Catchment id:", cat_id)
            # print("Catchment area:", cat_area)
            # print("N samples:", n_samp)

            stat_df = utils.get_nve_gts_api_aggregated_time_series(
                stn_gdf,
                pars,
                st_dt,
                end_dt,
                id_col=cat_id_col,
                n_samp=n_samp,
                random_state=42,
                all_pt_gdf=all_pt_gdf
            )

            stat_df["unit"] = stat_df["unit"].replace({"Celcius": "c"})
            stat_df["par"] = "gts_avg_" + stat_df["par"] + "_" + stat_df["unit"]

            stat_df = stat_df.set_index(
                ["datetime", "par"]
            )["value_mean"].unstack("par")

            stat_df.columns.name = ""

            stat_df = stat_df.resample("D").mean()

            stat_df["date"] = stat_df.index
            stat_df = stat_df.reset_index(drop=True)

            # Calculate modelled discharge from runoff
            if "gwb_q" in pars:
                # Only calculate discharge if the runoff download succeeded
                # (sometimes temp and precip download but runoff doesn't...)
                runoff_col = "gts_avg_gwb_q_mm"
                if runoff_col in stat_df.columns:
                    stat_df["gts_mod_discharge_m3ps"] = (
                        1e6 * cat_area * stat_df[runoff_col]
                        / (1000 * 24 * 60 * 60)
                    )
                else:
                    print(f"WARNING: No runoff data downloaded for {cat_id}")

            stat_df = stat_df.round(4)
            stat_df[cat_id_col] = cat_id

            stat_df.to_csv(csv_path, index=False)

            return f"{cat_id}: success"

        except URLError:
            retries += 1
            time.sleep(5)

    return f"{cat_id}: failed"


# Build gdf of points at grid cell centres on a 1 km grid
# Norway bounding box in EPSG 25833
xmin, ymin, xmax, ymax = -80000, 6449000, 1120000, 7945000
all_pt_df = nivapy.da.create_point_grid(xmin, ymin, xmax, ymax, 1000)
all_pt_gdf = gpd.GeoDataFrame(
    all_pt_df,
    geometry=gpd.points_from_xy(all_pt_df["x"], all_pt_df["y"], crs="epsg:25833"),
)

In [35]:
# Count how many catchments already have data downloaded for them,
# and how many are remaining

# API call parameters
out_folder = r"/home/jovyan/shared/common/norsink/vm_catchments_data"
temp_dir = os.path.join(out_folder, "gridded_by_site")

cat_id_col = "site_code"
cat_area_col = "area_km2"

catchment_ids = set(cat_gdf[cat_id_col].astype(str))

existing_ids = {
    os.path.splitext(os.path.basename(f))[0]
    for f in glob.glob(os.path.join(temp_dir, "*.csv"))
}

remaining_ids = list(catchment_ids - existing_ids)

print(
    f"Catchments: {len(catchment_ids):,} | "
    f"Processed: {len(existing_ids):,} | "
    f"Remaining: {len(remaining_ids):,}"
)

Catchments: 8,482 | Processed: 8,482 | Remaining: 0


In [36]:
# Threaded
# API call parameters
out_folder = r"/home/jovyan/shared/common/norsink/vm_catchments_data"
temp_dir = os.path.join(out_folder, "gridded_by_site/retry")

cat_id_col = "site_code"
cat_area_col = "area_km2"

# pars = ["tm", "rr", "gwb_q", "qtt"]
pars = ["gwb_q", "tm", "rr"] 
max_retries = 5
st_dt = '1980-01-01'
end_dt = '2025-12-31'

# Threading params
n_workers = 7

# Start processing
cats_to_processes_df = cat_gdf.loc[cat_gdf['site_code'].isin(redo_missing_col_sites)]
# cats_to_processes_df = cat_gdf.loc[cat_gdf['site_code'].isin(remaining_ids)]
# cats_to_process_df = cat_gdf

groups = list(cats_to_processes_df.groupby(cat_id_col))
t0 = perf_counter()

with ThreadPoolExecutor(max_workers=n_workers) as executor:

    futures = [
        executor.submit(process_catchment, cat_id, stn_gdf)
        for cat_id, stn_gdf in groups
    ]

    for future in as_completed(futures):
        print(future.result())

    elapsed = perf_counter() - t0
    print(
        f"max_workers={n_workers:2d} | "
        f"{elapsed/60:.1f} min ({elapsed:.0f} s)"
    )

Number of grid points: 160


Looping over parameters:   0%|          | 0/3 [00:00<?, ?it/s]

Looping over grid cells:   0%|          | 0/160 [00:00<?, ?it/s]

Number of grid points: 183


Looping over parameters:   0%|          | 0/3 [00:00<?, ?it/s]

...

Looping over grid cells:   0%|          | 0/196 [00:00<?, ?it/s]

Looping over grid cells:   0%|          | 0/195 [00:00<?, ?it/s]

Looping over grid cells:   0%|          | 0/202 [00:00<?, ?it/s]

Looping over grid cells:   0%|          | 0/196 [00:00<?, ?it/s]

012-28357: success
012-28228: success
012-28226: success
012-28522: success
012-28223: success
012-28371: success
012-28222: success
max_workers= 7 | 9.6 min (575 s)


## Without threading

Useful for testing/single sites

In [19]:
# # Output folder (result csvs added here)
# out_folder = r"/home/jovyan/shared/common/norsink/vm_catchments_data"
# temp_dir = os.path.join(out_folder, "gridded_by_site/retry")

# cat_id_col = "site_code"
# cat_area_col = "area_km2"

# # pars = ["tm", "rr", "gwb_q", "qtt"]
# # pars = ["gwb_q", "tm", "rr"]
# pars = ["tm", "rr"]
# # pars = ["gwb_q"]
# max_retries = 5
# st_dt = '1980-01-01'
# end_dt = '2025-12-31'

# # cat_gdf_remaining = cat_gdf.loc[cat_gdf['site_code'].isin(remaining_ids)]

# # Loop over catchments
# # for cat_id, stn_gdf in cat_gdf_remaining.groupby(cat_id_col):
# for cat_id, stn_gdf in cat_gdf.loc[cat_gdf[cat_id_col] == '012-28228'].groupby(cat_id_col):
    
#     print(cat_id)
    
#     # Skip if already processed
#     csv_path = os.path.join(temp_dir, f"{cat_id}.csv")
#     if os.path.isfile(csv_path):
#         # print("  Already processed.")
#         continue
    
#     retries = 0
    
#     while retries < max_retries:
#         try:
#             # print("  Processing.")
#             cat_area = stn_gdf.iloc[0][cat_area_col]
#             n_samp = get_sample_size(cat_area)
#             print("Catchment area:", cat_area)
#             print("Number of samples:", n_samp)

#             if st_dt < end_dt:
#                 stat_df = utils.get_nve_gts_api_aggregated_time_series(
#                     stn_gdf,
#                     pars,
#                     st_dt,
#                     end_dt,
#                     id_col=cat_id_col,
#                     n_samp=n_samp,
#                     random_state=42,
#                     all_pt_gdf = all_pt_gdf
#                 )
#                 stat_df["unit"] = stat_df["unit"].replace({"Celcius": "c"})
#                 stat_df["par"] = "gts_avg_" + stat_df["par"] + "_" + stat_df["unit"]
#                 stat_df = stat_df.set_index(["datetime", "par"])["value_mean"].unstack("par")
#                 stat_df.columns.name = ""
#                 stat_df = stat_df.resample("D").mean()
#                 stat_df["date"] = stat_df.index
#                 stat_df = stat_df.reset_index(drop=True)

#                 # Calculate modelled discharge from runoff
#                 if "gwb_q" in pars:
#                     # Only calculate discharge if the runoff download succeeded
#                     # (sometimes temp and precip download but runoff doesn't...)
#                     runoff_col = "gts_avg_gwb_q_mm"
#                     if runoff_col in stat_df.columns:
#                         stat_df["gts_mod_discharge_m3ps"] = (
#                             1e6 * cat_area * stat_df[runoff_col]
#                             / (1000 * 24 * 60 * 60)
#                         )
#                     else:
#                         print(f"WARNING: No runoff data downloaded for {cat_id}")

#                 stat_df = stat_df.round(4)  # 3 d.p. prob fine too

#                 # Add site_code
#                 stat_df[cat_id_col] = cat_id
#                 stat_df.to_csv(csv_path, index=False)
#             break
#         except URLError as e:
#             retries += 1
#             time.sleep(5)

012-28228
Catchment area: 1701.3770711455288
Number of samples: 183
Number of grid points: 183


Looping over parameters:   0%|          | 0/2 [00:00<?, ?it/s]

Looping over grid cells:   0%|          | 0/183 [00:00<?, ?it/s]

Looping over grid cells:   0%|          | 0/183 [00:00<?, ?it/s]

# QC

In [59]:
def check_file_availability(temp_dir, catch_ids):

    flist = glob.glob(os.path.join(temp_dir, "*.csv"))

    existing_ids = {
        os.path.splitext(os.path.basename(f))[0]
        for f in glob.glob(os.path.join(temp_dir, "*.csv"))
    }

    remaining_ids = list(catchment_ids - existing_ids)

    print(
        f"Catchments: {len(catchment_ids):,} | "
        f"Processed: {len(existing_ids):,} | "
        f"Remaining: {len(remaining_ids):,}"
    )


def qc_each_file(temp_dir, required_cols):

    flist = glob.glob(os.path.join(temp_dir, "*.csv"))
    summary = []

    for i, fpath in enumerate(flist):

        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1:,} of {len(flist):,} files")

        site = os.path.splitext(os.path.basename(fpath))[0]

        try:
            df = pd.read_csv(fpath)

            missing_cols = sorted(set(required_cols) - set(df.columns))
            has_required_cols = len(missing_cols) == 0

            if has_required_cols:
                nan_counts = df[required_cols].isna().sum()
                total_nans = int(nan_counts.sum())
                cols_with_nans = ",".join(
                    nan_counts[nan_counts > 0].index.tolist()
                )
            else:
                total_nans = None
                cols_with_nans = None

            summary.append({
                "site_code": site,
                "n_rows": len(df),
                "has_required_cols": has_required_cols,
                "missing_columns": ",".join(missing_cols) if missing_cols else "",
                "n_nans": total_nans,
                "columns_with_nans": cols_with_nans,
            })

        except Exception as e:
            summary.append({
                "site_code": site,
                "n_rows": None,
                "has_required_cols": False,
                "missing_columns": "READ_ERROR",
                "n_nans": None,
                "columns_with_nans": str(e),
            })

    # ------------------------------------------------------------------
    # Create summary table
    # ------------------------------------------------------------------
    qc_df = pd.DataFrame(summary)

    print("\nQC SUMMARY")
    print(f"Files checked                 : {len(qc_df):,}")
    print(f"Files with missing columns    : {(~qc_df['has_required_cols']).sum():,}")
    print(f"Files containing NaNs         : {(qc_df['n_nans'].fillna(0) > 0).sum():,}")
    print(f"Files that could not be read  : {(qc_df['missing_columns'] == 'READ_ERROR').sum():,}")

    return qc_df



## First pass

In [60]:
out_folder = r"/home/jovyan/shared/common/norsink/vm_catchments_data"
temp_dir = os.path.join(out_folder, "gridded_by_site")

# ------------------------------------------------------------------
# Check file availability
# ------------------------------------------------------------------
catchment_ids = set(cat_gdf[cat_id_col].astype(str))
check_file_availability(temp_dir, catchment_ids)

Catchments: 8,482 | Processed: 8,482 | Remaining: 0


In [9]:
# ------------------------------------------------------------------
# QC each file
# ------------------------------------------------------------------
# Required columns
required_cols = [
    "gts_avg_gwb_q_mm",
    "gts_avg_rr_mm",
    "gts_avg_tm_c",
    "date",
    "gts_mod_discharge_m3ps",
    "site_code",
]


summary = []

for i, fpath in enumerate(temp_dir):

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1:,} of {len(flist):,} files")

    site = os.path.splitext(os.path.basename(fpath))[0]

    try:
        df = pd.read_csv(fpath)

        missing_cols = sorted(set(required_cols) - set(df.columns))
        has_required_cols = len(missing_cols) == 0

        if has_required_cols:
            nan_counts = df[required_cols].isna().sum()
            total_nans = int(nan_counts.sum())
            cols_with_nans = ",".join(
                nan_counts[nan_counts > 0].index.tolist()
            )
        else:
            total_nans = None
            cols_with_nans = None

        summary.append({
            "site_code": site,
            "n_rows": len(df),
            "has_required_cols": has_required_cols,
            "missing_columns": ",".join(missing_cols) if missing_cols else "",
            "n_nans": total_nans,
            "columns_with_nans": cols_with_nans,
        })

    except Exception as e:
        summary.append({
            "site_code": site,
            "n_rows": None,
            "has_required_cols": False,
            "missing_columns": "READ_ERROR",
            "n_nans": None,
            "columns_with_nans": str(e),
        })

# ------------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------------

qc_df = pd.DataFrame(summary)

print("\nQC SUMMARY")
print(f"Files checked                 : {len(qc_df):,}")
print(f"Files with missing columns    : {(~qc_df['has_required_cols']).sum():,}")
print(f"Files containing NaNs         : {(qc_df['n_nans'].fillna(0) > 0).sum():,}")
print(f"Files that could not be read  : {(qc_df['missing_columns'] == 'READ_ERROR').sum():,}")

# Optional: save results
qc_df.to_csv(os.path.join(temp_dir, "gts-api_qc_summary.csv"), index=False)

qc_df

Catchments: 8,482 | Processed: 8,482 | Remaining: 0
Processed 500 of 8,482 files
Processed 1,000 of 8,482 files
Processed 1,500 of 8,482 files
Processed 2,000 of 8,482 files
Processed 2,500 of 8,482 files
Processed 3,000 of 8,482 files
Processed 3,500 of 8,482 files
Processed 4,000 of 8,482 files
Processed 4,500 of 8,482 files
Processed 5,000 of 8,482 files
Processed 5,500 of 8,482 files
Processed 6,000 of 8,482 files
Processed 6,500 of 8,482 files
Processed 7,000 of 8,482 files
Processed 7,500 of 8,482 files
Processed 8,000 of 8,482 files

QC SUMMARY
Files checked                 : 8,482
Files with missing columns    : 12
Files containing NaNs         : 8,470
Files that could not be read  : 0

Example problem files:
      site_code  n_rows  has_required_cols  \
186   311-83963   16802              False   
1122  012-28228   16802              False   
1160  009-42405   16802              False   
1932  085-87977   16802              False   
2258  012-28222   16802              False 

,site_code,n_rows,has_required_cols,missing_columns,n_nans,columns_with_nans
0,002-61839,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
1,016-123791,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
2,012-81581,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
3,086-50085,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
4,002-39293,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
...,...,...,...,...,...,...
8477,002-118868,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
8478,002-88975,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
8479,126-104804,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
8480,012-90094,16802,True,,8.0,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"


## Download succceeded but missing one or more parameters - set up for retry

Files with missing columns after first round of download: 12. Of these:
- Missing discharge: 4 sites. One of these we would expect data for (Vettrebekken, '009-42405', a small catchment on W side of Oslofjord. But it's on a narrow peninsula, perhaps classed as sea in HBV? Loose site with 20 d.p. if allow to drop. Retry failed.). Rest are all basically 100% outside Norway (see below), so expected to fail with Q (interesting that temp and precip succeeded). If want to keep these, will have to do some manual tweaking. E.g. use observed discharge or fill from elsewhere.

- Missing met data: 8:

    - These are almost all large catchments in 012 (Drammenselva). Odd, as there are about 66 large catchments here and the rest of them succeeded. Just a case of retrying?
    - Manually identified as wrong (and failed to get met data): '085-87977': catchment delineation in wrong place - outlet moved out to sea. Real catch much smaller. Drop.

After this exploration, see that all the sites that failed for Q it's expected (possible exception of 009-42405, but retry failed there). For met, a retry should work for 7/8 of the sites. So make a list and re-do the download for them. Find that on a second pass it works. End result is that successfully download met data for n-1 sites, and Q data for n-4 sites. i.e. 5 sites overall that don't have data for when merge the missing sites.

In [10]:
files_missing_cols = ~qc_df['has_required_cols']
files_missing_cols

,site_code,n_rows,has_required_cols,missing_columns,n_nans,columns_with_nans
186,311-83963,16802,False,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps",NaN,None
1122,012-28228,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None
1160,009-42405,16802,False,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps",NaN,None
1932,085-87977,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None
2258,012-28222,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None
3604,012-28223,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None
4610,314-62046,16802,False,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps",NaN,None
5584,012-28226,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None
5791,012-28371,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None
6234,012-28522,16802,False,"gts_avg_rr_mm,gts_avg_tm_c",NaN,None


In [13]:
# Look at whether it's due to being outside Norway
pt_dir = r"/home/jovyan/shared/common/norsink/spatial/monitored_chem_catchments"
pts_fpath = os.path.join(pt_dir, "outlet_locations_vm-rivs-quant-oc_v3-full.csv")
pts_df = pd.read_csv(pts_fpath)

pts_df.loc[pts_df['site_code'].isin(files_missing_cols['site_code'])]

,site_code,lon,lat,site_id,Name,start_year,end_year,n_years,n_samples,av_freq,pt_in_lake,area_km2,frac_sw,frac_fi,frac_ru,frac_outside_no
0,001-119928,11.653745,59.476520,119928,Oppstrøms Bommen RA,2011,2025,14.0,14,1.00,False,1008.505005,0.010104,0.0,0.0,0.010104
1,001-119929,11.654461,59.472300,119929,Nedstrøms Bommen RA,2011,2025,14.0,14,1.00,False,1009.065359,0.010099,0.0,0.0,0.010099
2,001-30653,11.437438,59.070299,30653,YSTE YSTEHEDEBEKKEN,1987,1987,1.0,4,4.00,False,6.702626,0.000000,0.0,0.0,0.000000
3,001-30683,11.295421,59.121185,30683,"Unnebergbekken, UNNE",1987,2016,29.1,5,0.17,False,15.372746,0.000000,0.0,0.0,0.000000
4,001-30687,11.528412,58.987892,30687,ØRB2 ØRBEKKEN,1991,1991,1.0,4,4.00,False,42.290760,0.000000,0.0,0.0,0.000000


In [34]:
# List of site codes where met data failed unexpectedly; then retry the download (just repurposing code higher up in NB)

expect_fail = ['085-87977']  # delineated catchment out at sea, drop

redo_missing_col_sites = files_missing_cols.loc[files_missing_cols['missing_columns'] == "gts_avg_rr_mm,gts_avg_tm_c"]['site_code']

redo_missing_col_sites = redo_missing_col_sites.loc[~redo_missing_col_sites.isin(expect_fail)]
redo_missing_col_sites

1122    012-28228
2258    012-28222
3604    012-28223
5584    012-28226
5791    012-28371
6234    012-28522
7994    012-28357
Name: site_code, dtype: object

In [41]:
# Recheck having now re-downloaded for these 7 sites. Successful? Yes :-)

out_folder = r"/home/jovyan/shared/common/norsink/vm_catchments_data"
temp_dir = os.path.join(out_folder, "gridded_by_site/retry")

cat_id_col = "site_code"

# Required columns
required_cols = [
    "gts_avg_gwb_q_mm",
    "gts_avg_rr_mm",
    "gts_avg_tm_c",
    "date",
    "gts_mod_discharge_m3ps",
    "site_code",
]

qc_retry_df = qc_each_file(temp_dir, required_cols)

qc_retry_df


QC SUMMARY
Files checked                 : 7
Files with missing columns    : 0
Files containing NaNs         : 7
Files that could not be read  : 0


,site_code,n_rows,has_required_cols,missing_columns,n_nans,columns_with_nans
0,012-28228,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
1,012-28222,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
2,012-28223,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
3,012-28226,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
4,012-28371,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
5,012-28522,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"
6,012-28357,16802,True,,8,"gts_avg_gwb_q_mm,gts_mod_discharge_m3ps"


## Why do all sites have 8 NaNs in discharge?

Having picked a couple of sites at random, see that these are always:
- 2022-06-17 and -18
- 2024-01-06 and -07

James confirmed he has those NaNs from separate workflows, so it's not something to do with e.g. the aggregation, but a real gap. Very odd.

i.e. need to fill the NaNs. Easiest is **linear interpolation**.
Doesn't seem worth doing now, instead do when making the master big table.

In [52]:
qc_df.describe()

,n_rows,n_nans
count,8482.0,8470.0
mean,16802.0,8.0
std,0.0,0.0
min,16802.0,8.0
25%,16802.0,8.0
50%,16802.0,8.0
75%,16802.0,8.0
max,16802.0,8.0


In [54]:
# Pick a couple of sites at random
temp_dir = os.path.join(out_folder, "gridded_by_site/retry")
test_fpath = os.path.join(temp_dir, "012-28228.csv")
test_df = pd.read_csv(test_fpath)
test_df.loc[test_df['gts_avg_gwb_q_mm'].isna()]

,gts_avg_gwb_q_mm,gts_avg_rr_mm,gts_avg_tm_c,date,gts_mod_discharge_m3ps,site_code
15508,NaN,0.0454,7.1027,2022-06-17,NaN,012-28228
15509,NaN,5.6672,7.2579,2022-06-18,NaN,012-28228
16076,NaN,0.0716,-22.5699,2024-01-06,NaN,012-28228
16077,NaN,0.0000,-19.2623,2024-01-07,NaN,012-28228


In [50]:
test_fpath = os.path.join(temp_dir, "012-28222.csv")
test_df = pd.read_csv(test_fpath)
test_df.loc[test_df['gts_avg_gwb_q_mm'].isna()]

,gts_avg_gwb_q_mm,gts_avg_rr_mm,gts_avg_tm_c,date,gts_mod_discharge_m3ps,site_code
15508,NaN,0.0332,9.1097,2022-06-17,NaN,012-28222
15509,NaN,4.4464,8.9561,2022-06-18,NaN,012-28222
16076,NaN,0.0255,-22.8760,2024-01-06,NaN,012-28222
16077,NaN,0.0000,-19.5316,2024-01-07,NaN,012-28222


In [51]:
temp_dir = os.path.join(out_folder, "gridded_by_site")
test_fpath = os.path.join(temp_dir, "016-123791.csv")
test_df = pd.read_csv(test_fpath)
test_df.loc[test_df['gts_avg_gwb_q_mm'].isna()]

,gts_avg_gwb_q_mm,gts_avg_rr_mm,gts_avg_tm_c,date,gts_mod_discharge_m3ps,site_code
15508,NaN,0.0328,10.7000,2022-06-17,NaN,016-123791
15509,NaN,6.4714,9.7076,2022-06-18,NaN,016-123791
16076,NaN,0.0000,-23.9361,2024-01-06,NaN,016-123791
16077,NaN,0.0000,-20.8244,2024-01-07,NaN,016-123791


# Combine all data

Optional. For ML, do this in the master compile data notebook (NB 04)

In [ ]:
# Combine all data
# flist = glob.glob(os.path.join(temp_dir, "*.csv"))
# df = pd.concat([pd.read_csv(fpath) for fpath in flist], axis="rows")
# df["date"] = pd.to_datetime(df["date"])

# Save
# catchment_set = "monitored-oc-catchs"  # for filenaming
# fname_suffix = f"daily_nvegts_temp-q_1980-2025"
# merged_csv_path = os.path.join(out_folder, f"{catchment_set}_{fname_suffix}.csv")
# df.to_csv(merged_csv_path, index=False)

# df